### Important!
#### First, create a MongoDB account, then set up a cluster, initialize your database, and obtain your MongoDB connection string. It should look like:

Now create a .env file and paste this *MONGODB_DRIVER_STRING* into that file.

### MongoClient Setup

In [6]:
import os
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()
# MONGODB_DRIVER_URI = MONGODB_DRIVER_STRING
MONGODB_DRIVER_URI = os.getenv('MONGODB_DRIVER_STRING')

# Connect to MongoDB
try:
    client = MongoClient(MONGODB_DRIVER_URI, serverSelectionTimeoutMS=5000)
    # This actually forces a connection
    client.admin.command("ping")
    print("✅ Connected to MongoDB successfully")
    
except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected to MongoDB successfully


In [4]:
db = client["database_embd"]
print("Databases:", client.list_database_names())
print("Collections:", db.list_collection_names())

ServerSelectionTimeoutError: ac-8jurarg-shard-00-01.zqqbm7m.mongodb.net:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 6a565cb1740c0e64ec2fa29b, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-8jurarg-shard-00-00.zqqbm7m.mongodb.net', 27017) server_type: Unknown, rtt: None>, <ServerDescription ('ac-8jurarg-shard-00-01.zqqbm7m.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('ac-8jurarg-shard-00-01.zqqbm7m.mongodb.net:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescription ('ac-8jurarg-shard-00-02.zqqbm7m.mongodb.net', 27017) server_type: Unknown, rtt: None>]>

### Load dataset

In [60]:
import pickle
with open("../../generate_embedding/embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

with open("../../../Flicker 30k hf/captions.txt", "r", encoding='utf-8') as f:
    captions = f.readlines()

In [61]:
print("Top-level fields in each record:")
for col, val in embeddings[0].items():
    print(f"{col}: {type(val).__name__}")

Top-level fields in each record:
image_id: str
embedding: ndarray


### Get captions

In [62]:
print(type(captions), captions[1])

<class 'list'> 1000092795.jpg, Two young guys with shaggy hair look at their hands while hanging out in the yard .



In [ ]:
import re
from collections import defaultdict

def clean_text(text):
    text = re.sub(r'["\']', '', text) # no quotes(eg: ', ")
    text = re.sub(r'^[^a-zA-Z0-9]+', '', text) # starts with alphanumeric
    text = re.sub(r'\s+\.', '.', text) # before after space
    return text.rstrip()

caption_dict = defaultdict(list)

for caption in captions[1:]:
    id, text = caption.split('.jpg')
    image_id = id + '.jpg'
    caption = clean_text(text)
    caption_dict[image_id].append(caption)

In [64]:
print(len(caption_dict))
caption_dict['1000092795.jpg']

31783


['Two young guys with shaggy hair look at their hands while hanging out in the yard.',
 'Two young , White males are outside near many bushes.',
 'Two men in green shirts are standing in a yard.',
 'A man in a blue shirt standing in a garden.',
 'Two friends enjoy time spent together.']

## Upload to remote

In [69]:
db = client["database_embd"]
print("Databases:", client.list_database_names())
print("Collections:", db.list_collection_names())

Databases: ['database_embd', 'embeddings', 'admin', 'local']
Collections: ['embeddings_31k']


In [ ]:
# Try with sample
sample = embeddings[5]  # take first record

# Flexible Schema
doc = {
    "image_id": sample["image_id"],
    "image_embedding": sample["embedding"].tolist(),  # IMPORTANT
    "captions": caption_dict[sample["image_id"]] # Extracting only text strings
}

# Connect the remote cluster
client = MongoClient(MONGODB_DRIVER_URI)
collection = client['database_embd']['embeddings_31k']

# Insert One document to remote cluster
collection.insert_one(doc)

print("✅ Test insert successful")

✅ Test insert successful


print(collection.delete_many({}))

In [ ]:
print(collection.count_documents({}))

0


In [137]:
import os
import pickle
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()

# Config
DB_NAME = "database_embd"
COLLECTION_NAME = "embeddings_31k"
MONGO_URI = os.getenv('MONGODB_DRIVER_STRING')

EMBEDDING_PATH = "../../generate_embedding/embeddings.pkl"
CAPION_PATH = "../../../Flicker 30k hf/captions.txt"

BATCH_SIZE = 200
CHECKPOINT_FILE = "upload_checkpoint.txt"

import pickle
with open(EMBEDDING_PATH, "rb") as f:
    embeddings = pickle.load(f)

with open(CAPION_PATH, "r", encoding='utf-8') as f:
    captions = f.readlines()

# Load CheckPoint
start_idx = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        start_idx = int(f.read().strip())
    print(f"🔁 Resuming from index: {start_idx}")

# Connect to remote cluster
client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]

print("Databases:", client.list_database_names())
print("Collections:", db.list_collection_names())
print(f"\nCurrent DB: '{DB_NAME}', Collection: '{COLLECTION_NAME}'")

# Upload
batch = []
count = start_idx

try:
    for i in range(start_idx, len(embeddings)): #len(embeddings)
        item = embeddings[i]  # take first record

        # Flexible Schema
        doc = {
            "image_id": item["image_id"],
            "image_embedding": item["embedding"].tolist(),  # IMPORTANT
            "captions": caption_dict[item["image_id"]] # Extracting only text strings
        }

        batch.append(doc)

        if len(batch) == BATCH_SIZE:
            collection.insert_many(batch, ordered=False)  # fast
            count += len(batch)

            print(f"✅ Uploaded: {count}")

            # save checkpoint
            with open(CHECKPOINT_FILE, "w") as f:
                f.write(str(i + 1))

            batch = []

    # remaining
    if batch:
        collection.insert_many(batch, ordered=False)
        count += len(batch)

    print(f"🎉 DONE. Total uploaded: {count}")

except Exception as e:
    print(f"❌ Failed at index {i}: {e}")
    print(f"💾 Progress saved. Restart script to resume.")

🔁 Resuming from index: 29200
Databases: ['database_embd', 'embeddings', 'admin', 'local']
Collections: ['embeddings_31k']

Current DB: 'database_embd', Collection: 'embeddings_31k'
✅ Uploaded: 29400
✅ Uploaded: 29600
✅ Uploaded: 29800
✅ Uploaded: 30000
✅ Uploaded: 30200
✅ Uploaded: 30400
✅ Uploaded: 30600
✅ Uploaded: 30800
✅ Uploaded: 31000
✅ Uploaded: 31200
✅ Uploaded: 31400
✅ Uploaded: 31600
🎉 DONE. Total uploaded: 31783


### Less space in MongoDB, Reduce size!
If you want to reduce the size by half then we can use this, maybe be you can see a slight different in performance.


item["image_embedding"].astype("float32").tolist()

### Check for duplicates

In [144]:
total = collection.count_documents({})
distinct = len(collection.distinct("image_id"))

print(total, distinct)

31783 31783


## Important!
### Now go to MongoDB account and create two index

**Vector Index** : for vector search
**Text Index** : for text search

In [79]:
# Go to the MongoDB collecton paste this when creating two indivisual index

# Text Index
"""
{
  "mappings": {
    "dynamic": false,
    "fields": {
      "captions": {
        "type": "string"
      }
    }
  }
}"""

# Vector Index
"""{
  "fields": [
    {
      "type": "vector",
      "path": "image_embedding",
      "numDimensions": 512,
      "similarity": "cosine"
    }
  ]
}"""

'{\n  "fields": [\n    {\n      "type": "vector",\n      "path": "image_embedding",\n      "numDimensions": 512,\n      "similarity": "cosine"\n    }\n  ]\n}'

### Vector Index Search

In [1]:
import os
import pickle
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()

# Config
DB_NAME = "database_embd"
COLLECTION_NAME = "embeddings_31k"
MONGO_URI = os.getenv('MONGODB_DRIVER_STRING')

EMBEDDING_PATH = "../../generate_embedding/embeddings.pkl"
CAPION_PATH = "../../../Flicker 30k hf/captions.txt"

# MongoDB collection
client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]
print(f"\nCurrent DB: '{DB_NAME}', Collection: '{COLLECTION_NAME}'")

# Load the data
import pickle
with open(EMBEDDING_PATH, "rb") as f:
    embeddings = pickle.load(f)

with open(CAPION_PATH, "r", encoding='utf-8') as f:
    captions = f.readlines()

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\pymongo\pyopenssl_context.py:348: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  _crypto.X509.from_cryptography(x509.load_der_x509_certificate(cert))



Current DB: 'database_embd', Collection: 'embeddings_31k'


In [109]:
image_id = embeddings[0]['image_id']
query_emb = embeddings[0]['embedding'].tolist()
captions = caption_dict[image_id]

In [116]:
res = collection.aggregate([{
    "$vectorSearch": {
      "index": "vector_index",
      "path": "image_embedding",
      "queryVector": query_emb,
      "numCandidates": 100,
      "limit": 10
    }
  }
])

In [118]:
# MongoDB Text Search
def text_search(query_text, limit=10):
    pipeline = [
        {
            "$search": {
                "index": "default",
                "text": {
                    "query": query_text,
                    "path": "captions"   # ✅ updated
                }
            }
        },
        {
            "$project": {
                "_id": 0,
                "image_id": 1,
                "captions": 1,
                "score": {"$meta": "searchScore"}
            }
        },
        {
            "$limit": limit
        }
    ]
    
    return list(collection.aggregate(pipeline))

# MongoDB Vector Search
def vector_search(query_embedding, top_k=10, page=1):
    skip = (page - 1) * top_k

    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "image_embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": top_k + skip,  # fetch extra
            }
        },
        {"$skip": skip},  # pagination
        {"$limit": top_k},
        {
            "$project": {
                "_id": 0,
                "image_id": 1,
                "captions": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]

    return list(collection.aggregate(pipeline))

### Now test the indexes

In [ ]:
image_id = embeddings[0]['image_id']
captions = caption_dict[image_id]
query_emb = embeddings[0]['embedding'].tolist()

In [ ]:
try:
    # Use the first caption from the first item as a realistic query
    sample_text = captions[3]
    print(f"\n[Normal] Searching for text: '{sample_text}'")
    
    res_text = text_search(sample_text, limit=10)
    print(f"Results found: {len(res_text)}")
    if res_text:
        print(f"Top score: {res_text[0].get('score')} | Captions: {res_text[0].get('captions')[:1]}")
except Exception as e:
    print(f"Error: {e}")


[Normal] Searching for text: 'A man in a blue shirt standing in a garden.'
Results found: 5
Top score: 2.3751726150512695 | Captions: ['Two young guys with shaggy hair look at their hands while hanging out in the yard.']


In [126]:
try:
    print("\n[Edge Case] Empty string ''")
    res_empty = text_search("", limit=3)
    print(f"Results found: {len(res_empty)}")
except Exception as e:
    print(f"Error expected and caught: {e}")


[Edge Case] Empty string ''
Error expected and caught: "text.query" cannot be empty, full error: {'ok': 0.0, 'errmsg': '"text.query" cannot be empty', 'code': 8, 'codeName': 'UnknownError', '$clusterTime': {'clusterTime': Timestamp(1783327208, 2), 'signature': {'hash': b'\xad\xde\x07\xa3\xcdBI+\xc2\xbe\xdb\x95*\xd9\xef\xdc\x8c|s\x03', 'keyId': 7616462288513400837}}, 'operationTime': Timestamp(1783327208, 2)}


In [124]:
# 3. Edge Case: Deep Pagination
query_emb = embeddings[0]['embedding'].tolist()
try:
    print("\n[Edge Case] Deep pagination (page=10)")
    # Note: Vector search typically limits `$skip` internally unless configured
    res_page = vector_search(query_emb, top_k=5, page=10)
    print(f"Results found: {len(res_page)}")
    if res_page:
        print(f"Top score on page 10: {res_page[0].get('score')}")
except Exception as e:
    print(f"Error expected and caught: {e}")


[Edge Case] Deep pagination (page=10)
Results found: 0


In [125]:
# 4. Edge Case: Gibberish / Non-existent Word
try:
    print("\n[Edge Case] Gibberish query 'xyzqwe123456'")
    res_gibberish = text_search("xyzqwe123456", limit=3)
    print(f"Results found: {len(res_gibberish)}")
except Exception as e:
    print(f"Error expected and caught: {e}")


[Edge Case] Gibberish query 'xyzqwe123456'
Results found: 0


### Image Index Testing

In [127]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# 1. Setup Query
query_emb = embeddings[0]['embedding'].tolist()

# 2. Perform vector search matching the query
top_k = 5
res_vec = vector_search(query_emb, top_k=top_k, page=1)
print(f"For image {embeddings[0]['image_id']}, Found top {len(res_vec)} match\n")

For image 1000092795.jpg, Found top 5 match



In [129]:
from PIL import Image
from IPython.display import display

IMAGE_FOLDER = "../../../Flicker 30k hf/Images/chunk_1/"

print("Original Image")
img = Image.open(IMAGE_FOLDER + embeddings[0]['image_id'])  # fixed
display(img)

for res in res_vec:
    print(res['image_url'], res['captions'][0])
    img = Image.open(IMAGE_FOLDER + res['image_url'])  # fixed
    display(img)

Original Image


FileNotFoundError: [Errno 2] No such file or directory: '../../../Flicker 30k hf/Images/chunk_1/1000092795.jpg'